# DQN PER

## Pruebas

In [ ]:
import sys
import importlib

import numpy as np

sys.path.append("../src")

import replay_buffer
importlib.reload(replay_buffer)

from replay_buffer import SumTree

arbol_prueba = SumTree(capacidad=4)

prioridades = [1.0, 2.0, 3.0, 4.0]

for indice, prioridad in enumerate(prioridades):
    arbol_prueba.actualizar(
        indice_dato=indice,
        prioridad=prioridad,
    )

print(
    f"Prioridad total esperada: "
    f"{sum(prioridades):.2f}"
)
print(
    f"Prioridad total obtenida: "
    f"{arbol_prueba.prioridad_total:.2f}"
)

print("\nMUESTREO POR VALOR ACUMULADO")

valores_prueba = [0.5, 1.5, 4.5, 8.0]

for valor in valores_prueba:
    indice, prioridad = arbol_prueba.obtener(valor)

    print(
        f"Valor {valor:>4.1f} -> "
        f"índice {indice}, "
        f"prioridad {prioridad:.1f}"
    )

arbol_prueba.actualizar(
    indice_dato=0,
    prioridad=5.0,
)

print(
    "\nPrioridad total después de cambiar "
    f"la prioridad 0 de 1 a 5: "
    f"{arbol_prueba.prioridad_total:.2f}"
)

assert np.isclose(
    arbol_prueba.prioridad_total,
    14.0,
)

print("\nEl SumTree funciona correctamente.")

Prioridad total esperada: 10.00
Prioridad total obtenida: 10.00

MUESTREO POR VALOR ACUMULADO
Valor  0.5 -> índice 0, prioridad 1.0
Valor  1.5 -> índice 1, prioridad 2.0
Valor  4.5 -> índice 2, prioridad 3.0
Valor  8.0 -> índice 3, prioridad 4.0

Prioridad total después de cambiar la prioridad 0 de 1 a 5: 14.00

El SumTree funciona correctamente.


In [ ]:
import torch

importlib.reload(replay_buffer)

from replay_buffer import (
    SumTree,
    PrioritizedReplayBuffer,
)

buffer_per_prueba = PrioritizedReplayBuffer(
    capacidad=8,
    forma_observacion=(4, 84, 84),
    seed=42,
    alpha=0.6,
    epsilon_per=1e-6,
)

for indice in range(8):
    observacion = np.full(
        (4, 84, 84),
        fill_value=indice,
        dtype=np.uint8,
    )

    siguiente_observacion = np.full(
        (4, 84, 84),
        fill_value=indice + 1,
        dtype=np.uint8,
    )

    buffer_per_prueba.agregar(
        observacion=observacion,
        accion=indice % 6,
        recompensa=float(indice),
        siguiente_observacion=siguiente_observacion,
        finalizado=False,
    )

errores_td = np.array(
    [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 10.0],
    dtype=np.float32,
)

buffer_per_prueba.actualizar_prioridades(
    indices=np.arange(8),
    errores_td=errores_td,
)

conteos = np.zeros(8, dtype=np.int64)

for _ in range(500):
    (
        observaciones,
        acciones,
        recompensas,
        siguientes_observaciones,
        finalizados,
        indices,
        pesos,
    ) = buffer_per_prueba.muestrear(
        batch_size=4,
        device=torch.device("cpu"),
        beta=0.4,
    )

    for indice in indices:
        conteos[indice] += 1

print("FRECUENCIA DE MUESTREO")

for indice, conteo in enumerate(conteos):
    print(
        f"Experiencia {indice}: "
        f"{conteo} apariciones"
    )

print(
    "\nPrioridad total: "
    f"{buffer_per_prueba.arbol_prioridades.prioridad_total:.4f}"
)

print(
    "Forma de los pesos: "
    f"{pesos.shape}"
)

print(
    "Rango de los pesos: "
    f"{pesos.min().item():.4f} - "
    f"{pesos.max().item():.4f}"
)

assert len(buffer_per_prueba) == 8
assert pesos.shape == (4,)
assert torch.isfinite(pesos).all()
assert pesos.max() <= 1.0 + 1e-6

assert conteos[7] > conteos[:7].max()

print(
    "\nLa experiencia con mayor error TD fue "
    "muestreada con mayor frecuencia."
)

FRECUENCIA DE MUESTREO
Experiencia 0: 97 apariciones
Experiencia 1: 89 apariciones
Experiencia 2: 80 apariciones
Experiencia 3: 85 apariciones
Experiencia 4: 90 apariciones
Experiencia 5: 79 apariciones
Experiencia 6: 79 apariciones
Experiencia 7: 1401 apariciones

Prioridad total: 5.7394
Forma de los pesos: torch.Size([4])
Rango de los pesos: 0.3311 - 1.0000

La experiencia con mayor error TD fue muestreada con mayor frecuencia.


In [ ]:
from copy import deepcopy

import models
import train

importlib.reload(models)
importlib.reload(train)

from models import DQN
from train import (
    ConfigDQN,
    actualizar_modelo,
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

config_prueba_per = ConfigDQN(
    batch_size=32,
    capacidad_buffer=64,
    usar_per=True,
    per_alpha=0.6,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=1_000,
)

modelo_online_prueba = DQN(
    n_acciones=6
).to(device)

modelo_target_prueba = deepcopy(
    modelo_online_prueba
).to(device)

modelo_target_prueba.eval()

optimizador_prueba = torch.optim.Adam(
    modelo_online_prueba.parameters(),
    lr=config_prueba_per.learning_rate,
)

buffer_actualizacion_per = PrioritizedReplayBuffer(
    capacidad=64,
    forma_observacion=(4, 84, 84),
    seed=42,
    alpha=config_prueba_per.per_alpha,
    epsilon_per=config_prueba_per.per_epsilon,
)

rng_prueba = np.random.default_rng(42)

for _ in range(64):
    observacion = rng_prueba.integers(
        0,
        256,
        size=(4, 84, 84),
        dtype=np.uint8,
    )

    siguiente_observacion = rng_prueba.integers(
        0,
        256,
        size=(4, 84, 84),
        dtype=np.uint8,
    )

    buffer_actualizacion_per.agregar(
        observacion=observacion,
        accion=int(rng_prueba.integers(6)),
        recompensa=float(
            rng_prueba.choice([-1.0, 0.0, 1.0])
        ),
        siguiente_observacion=siguiente_observacion,
        finalizado=bool(
            rng_prueba.random() < 0.1
        ),
    )

inicio_hojas = (
    buffer_actualizacion_per.capacidad - 1
)

prioridades_antes = (
    buffer_actualizacion_per
    .arbol_prioridades
    .arbol[
        inicio_hojas:
        inicio_hojas + len(buffer_actualizacion_per)
    ]
    .copy()
)

metricas_prueba_per = actualizar_modelo(
    modelo_online=modelo_online_prueba,
    modelo_target=modelo_target_prueba,
    replay_buffer=buffer_actualizacion_per,
    optimizador=optimizador_prueba,
    config=config_prueba_per,
    device=device,
    usar_double_dqn=False,
    paso_global=500,
)

prioridades_despues = (
    buffer_actualizacion_per
    .arbol_prioridades
    .arbol[
        inicio_hojas:
        inicio_hojas + len(buffer_actualizacion_per)
    ]
    .copy()
)

prioridades_modificadas = np.count_nonzero(
    ~np.isclose(
        prioridades_antes,
        prioridades_despues,
    )
)

print(f"Dispositivo: {device}")
print(
    f"Loss: "
    f"{metricas_prueba_per['loss']:.6f}"
)
print(
    f"Error TD promedio: "
    f"{metricas_prueba_per['error_td_promedio']:.6f}"
)
print(
    f"Beta utilizado: "
    f"{metricas_prueba_per['beta_per']:.3f}"
)
print(
    f"Peso de importancia promedio: "
    f"{metricas_prueba_per['peso_importancia_promedio']:.3f}"
)
print(
    f"Prioridades modificadas: "
    f"{prioridades_modificadas}"
)

assert np.isfinite(
    metricas_prueba_per["loss"]
)

assert prioridades_modificadas > 0

assert np.isclose(
    metricas_prueba_per["beta_per"],
    0.7,
)

print(
    "\nLa actualización con PER funciona correctamente."
)

Dispositivo: mps
Loss: 0.214474
Error TD promedio: 0.449528
Beta utilizado: 0.700
Peso de importancia promedio: 1.000
Prioridades modificadas: 32

La actualización con PER funciona correctamente.


In [4]:
import gc
import pandas as pd
from pathlib import Path

importlib.reload(train)

from train import (
    ConfigDQN,
    entrenar_dqn,
)

objetos_temporales = [
    "buffer_per_prueba",
    "buffer_actualizacion_per",
    "modelo_online_prueba",
    "modelo_target_prueba",
    "optimizador_prueba",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_smoke_per = ConfigDQN(
    nombre_experimento="smoke_test_dqn_per",
    total_pasos=2_000,

    usar_per=True,
    per_alpha=0.6,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=2_000,
    per_epsilon=1e-6,

    capacidad_buffer=2_000,
    inicio_entrenamiento=200,
    batch_size=32,
    frecuencia_entrenamiento=4,
    frecuencia_actualizacion_target=500,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=1_000,

    frecuencia_log=250,
    frecuencia_evaluacion=2_000,
    episodios_evaluacion=1,
)

resultado_smoke_per = entrenar_dqn(
    config=config_smoke_per,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print("\nSMOKE TEST DQN + PER FINALIZADO")
print(
    "Mejor promedio de evaluación: "
    f"{resultado_smoke_per['mejor_promedio_evaluacion']:.2f}"
)

ruta_updates_smoke = Path(
    "../logs/entrenamientos/"
    "smoke_test_dqn_per/actualizaciones.csv"
)

df_updates_smoke = pd.read_csv(
    ruta_updates_smoke
)

display(
    df_updates_smoke[
        [
            "paso_global",
            "loss",
            "error_td_promedio",
            "beta_per",
            "peso_importancia_promedio",
            "tamano_buffer",
        ]
    ]
)

A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Paso 250/2,000 | episodio=1 | epsilon=0.775 | loss=0.0054 | Q=0.071
Paso 500/2,000 | episodio=2 | epsilon=0.550 | loss=0.0133 | Q=0.103
Paso 750/2,000 | episodio=3 | epsilon=0.325 | loss=0.0021 | Q=0.185
Paso 1,000/2,000 | episodio=5 | epsilon=0.100 | loss=0.0039 | Q=0.177
Paso 1,250/2,000 | episodio=6 | epsilon=0.100 | loss=0.0104 | Q=0.297
Paso 1,500/2,000 | episodio=7 | epsilon=0.100 | loss=0.0026 | Q=0.246
Paso 1,750/2,000 | episodio=9 | epsilon=0.100 | loss=0.0049 | Q=0.373
Paso 2,000/2,000 | episodio=10 | epsilon=0.100 | loss=0.0013 | Q=0.336

EVALUACIÓN | paso=2,000 | promedio=80.00 | mediana=80.00 | máximo=80.00


SMOKE TEST DQN + PER FINALIZADO
Mejor promedio de evaluación: 80.00


,paso_global,loss,error_td_promedio,beta_per,peso_importancia_promedio,tamano_buffer
0,250,0.005354,0.083589,0.4744,0.453879,250
1,500,0.013343,0.214829,0.5500,0.373648,500
2,750,0.002121,0.115667,0.6244,0.180380,750
3,1000,0.003941,0.129842,0.7000,0.312108,1000
4,1250,0.010394,0.227189,0.7744,0.377781,1250
5,1500,0.002597,0.148887,0.8500,0.194636,1500
6,1750,0.004911,0.191007,0.9244,0.289631,1750
7,2000,0.001277,0.171572,1.0000,0.159696,2000


## Entrenamiento DQN + PER